# Phase 3 : Ingénierie des Caractéristiques (Feature Engineering)

Ce notebook présente la création des caractéristiques prédictives pour nos modèles de Machine Learning et Deep Learning, et prépare les jeux d'entraînement et de test.

## Caractéristiques créées :
1. **Retards (Lags)** : 1, 2, 6, et 12 mois sur la variable cible des arrivées.
2. **Moyennes Mobiles (Rolling Means) et Volatilité** : Calculées sur 3, 6, et 12 mois en décalant de 1 pour éviter toute fuite d'information (*data leakage*).
3. **Taux de Croissance Annuel (YoY)** : Calculé de manière décalée pour être prédictif.
4. **Composants Temporels Cycliques** : Encodage trigonométrique du mois (`sin_month`, `cos_month`) et trimestres.
5. **Saisonnalité catégorielle** : Encodage One-Hot des saisons (Hiver, Printemps, Été, Automne).
6. **Variables Macroéconomiques Retardées** : Retard de 1 mois et 3 mois pour le REER, les prix du pétrole, FDI et taux de pauvreté.
7. **Dummies d'Événements** : Flag COVID-19 et flag Coupe du Monde (CDM) 2026/2030.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np

from src.config import DATA_DIR, TARGET_COL, TRAIN_END_DATE, TEST_START_DATE
import src.features as feat

## 1. Chargement des Données Finales Nettoyées

In [ ]:
input_path = os.path.join(DATA_DIR, 'merged_tourism_data_final.csv')
df_clean = pd.read_csv(input_path)
df_clean['Date'] = pd.to_datetime(df_clean['Date'])
print(f"Dimensions du dataset chargé : {df_clean.shape}")
df_clean.head()

## 2. Génération des Caractéristiques

Nous faisons appel au module `src.features` pour générer toutes nos caractéristiques temporelles, décalées et catégorielles d'un seul coup, sans risque de fuite de données.

In [ ]:
df_featured = feat.build_features(df_clean)
features_list = feat.get_feature_list()

print(f"Nombre de caractéristiques générées : {len(features_list)}")
print(f"Dimensions finales après feature engineering : {df_featured.shape}")
df_featured[features_list + ['Date', TARGET_COL]].tail()

## 3. Sauvegarde du Dataset avec Features

Sauvegarde pour référence globale de la modélisation.

In [ ]:
output_featured_path = os.path.join(DATA_DIR, 'DATASET_ML_FEATURES.csv')
df_featured.to_csv(output_featured_path, index=False)
print(f"Sauvegarde du dataset de features terminée -> {output_featured_path}")

## 4. Découpage Temporel (Train / Test Split)

Pour évaluer correctement les prévisions temporelles, nous ne devons pas mélanger les observations au hasard. Nous divisons donc chronologiquement :
- **Train Set** : Données antérieures ou égales au `{TRAIN_END_DATE}` (généralement fin 2022).
- **Test Set** : Données postérieures ou égales au `{TEST_START_DATE}` (généralement début 2023).

In [ ]:
# On élimine les lignes initiales où la cible Arrivals est vide
df_ml = df_featured.dropna(subset=[TARGET_COL]).copy()

# Validation des caractéristiques disponibles
valid_features = [c for c in features_list if c in df_ml.columns]

# Division temporelle
X_train = df_ml[df_ml['Date'] <= TRAIN_END_DATE][valid_features]
y_train = df_ml[df_ml['Date'] <= TRAIN_END_DATE][TARGET_COL]

X_test = df_ml[df_ml['Date'] >= TEST_START_DATE][valid_features]
y_test = df_ml[df_ml['Date'] >= TEST_START_DATE][TARGET_COL]

print(f"Taille du Train Set (1995 - {TRAIN_END_DATE}) : {X_train.shape[0]} mois")
print(f"Taille du Test Set ({TEST_START_DATE} - 2026) : {X_test.shape[0]} mois")

## 5. Export des Ensembles Séparés

Nous sauvegardons les ensembles de données séparés dans le sous-dossier `separted` (en conservant ce nom pour compatibilité avec l'existant).

In [ ]:
separated_dir = os.path.join(DATA_DIR, 'separted')
os.makedirs(separated_dir, exist_ok=True)

X_train.to_csv(os.path.join(separated_dir, 'X_train.csv'), index=False)
y_train.to_csv(os.path.join(separated_dir, 'y_train.csv'), index=False)
X_test.to_csv(os.path.join(separated_dir, 'X_test.csv'), index=False)
y_test.to_csv(os.path.join(separated_dir, 'y_test.csv'), index=False)

print(f"Sauvegarde des jeux de données d'entraînement et de test terminée dans : {separated_dir}")